In [5]:
import chromadb
import os
from sentence_transformers import SentenceTransformer
from ollama import Client

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query = "QTR 04 2025 revenue"

client = chromadb.HttpClient(
    host="localhost",
    port=8010
)

query_embedding = model.encode(
    query,
    normalize_embeddings=True
).tolist()

collection = client.get_or_create_collection(
    name="pdf_chunking"
)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

aw_docs = results.get('documents', [[]])
flattened_docs = [str(doc) for sublist in aw_docs for doc in sublist]
context_text = "\n".join(flattened_docs[:3])
print(context_text)

# Explicit connection (Optional, defaults to localhost)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4622.84it/s]



        Record quarterly revenue of $39.3 billion, up 12% from Q3 and up 78% from a year ago
        

        Record quarterly Data Center revenue of $35.6 billion, up 16% from Q3 and up 93% from a year ago
        

        For fiscal 2025, revenue was $130.5 billion, up 114% from a year ago. GAAP earnings per diluted share was $2.94, up 147% from a year ago. Non-GAAP earnings per diluted share was $2.99, up 130% from a year ago.
        


In [6]:
ollama_host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
client = Client(host=ollama_host)
prompt=f"query= {query},  context = {context_text}"

stream = client.generate(
    model="qwen2.5:1.5b",  # High-speed CPU optimized model
    prompt=prompt,
    stream=True,           # Enables immediate rendering loop
    options={
        "num_ctx": 4096,   # Low but safe window buffer for CPU RAM
        "temperature": 0.0 # Lowest setting reduces CPU computation load
    }
)

print("Assistant: ", end="", flush=True)
for chunk in stream:
    print(chunk['response'], end="", flush=True)

Assistant: The quarterly revenue for QTR 04 2025 is $39.3 billion, which represents an increase of 12% compared to the previous quarter (Q3) and an increase of 78% compared to a year ago. The data center revenue in QTR 04 2025 was $35.6 billion, showing a growth of 16% from the third quarter and 93% from a year earlier. For fiscal 2025 as a whole, the total revenue reached $130.5 billion, marking an increase of 114% over the previous year's figures. The GAAP earnings per diluted share for fiscal 2025 were $2.94, representing a rise of 147% from the same period in 2024. Additionally, non-GAAP earnings per diluted share for fiscal 2025 were $2.99, indicating an increase of 130% compared to the previous year's results.